In [0]:
from pyspark.sql.functions import col, when, lpad, concat, lit, to_timestamp, substring, try_to_date, coalesce, year, month, dayofmonth, dayofweek, hour, minute, to_utc_timestamp

print("1. Reading 28.9 Million Rows from Bronze Layer...")
bronze_flights = spark.table("aviation_project.bronze_historical_flights")

In [0]:
# ---------------------------------------------------------
# Part 1: Filtering and COVID Anomaly Removal
# ---------------------------------------------------------
print("2. Filtering for LIT flights and removing COVID anomaly (2020-2021)...")

filtered_flights = bronze_flights.filter(
    ((col("Origin") == "LIT") | (col("Dest") == "LIT")) & 
    (~col("Year").cast("int").isin(2020, 2021))
)

In [0]:
# ---------------------------------------------------------
# Part 2: Local Timestamp Reconstruction
# ---------------------------------------------------------
print("3. Reconstructing Local Timestamps...")

# Handle the new data dictionary format "yyyymmdd" defensively
clean_date = coalesce(
    try_to_date(col("FlightDate").cast("string"), "yyyy-MM-dd"),
    try_to_date(col("FlightDate").cast("string"), "yyyyMMdd"),
    try_to_date(col("FlightDate").cast("string"), "M/d/yyyy") 
)
date_str = clean_date.cast("string")

padded_time = lpad(col("CRSDepTime").cast("string"), 4, "0")
hour_str = substring(padded_time, 1, 2)
minute_str = substring(padded_time, 3, 2)

timestamp_str = concat(date_str, lit(" "), hour_str, lit(":"), minute_str, lit(":00"))

df_with_local_ts = filtered_flights.withColumn("local_scheduled_time", to_timestamp(timestamp_str, "yyyy-MM-dd HH:mm:ss"))
df_with_local_ts = df_with_local_ts.dropna(subset=["local_scheduled_time"])

In [0]:
# ---------------------------------------------------------
# Part 3: Feature Extraction, UTC Shift, & Multi-Class Target
# ---------------------------------------------------------
print("4. Extracting Features, Shifting to UTC, & Mapping Targets...")

flight_timezone_expr = when(col("Origin").isin("ATL", "CLT", "LGA", "DCA", "MIA"), "America/New_York") \
                .when(col("Origin").isin("LIT", "DFW", "ORD", "IAH", "DAL", "STL"), "America/Chicago") \
                .when(col("Origin") == "DEN", "America/Denver") \
                .when(col("Origin") == "LAS", "America/Los_Angeles") \
                .otherwise("UTC")

silver_flights = df_with_local_ts.select(
    # 1. Chronological Anchor
    to_utc_timestamp(col("local_scheduled_time"), flight_timezone_expr).alias("scheduled_time_utc"),
    
    # 2. Temporal Features
    year(col("local_scheduled_time")).alias("year"),
    month(col("local_scheduled_time")).alias("month"),
    dayofmonth(col("local_scheduled_time")).alias("day_of_month"),
    dayofweek(col("local_scheduled_time")).alias("day_of_week"),
    hour(col("local_scheduled_time")).alias("hour"),
    minute(col("local_scheduled_time")).alias("minute"),
    
    # 3. Flight Identifiers & Operations
    col("Operating_Airline").alias("airline_code"),
    col("Flight_Number_Operating_Airline").cast("string").alias("flight_number"),
    col("Distance").cast("double").alias("distance_miles"),
    
    # 4. Routing
    col("Origin").alias("origin_airport"),
    col("Dest").alias("destination_airport"),
    
    # 5. ML Targets (Multi-Class)
    coalesce(col("DepDelay").cast("int"), lit(0)).alias("delay_minutes"),
    
    # Flight Status: 0=On Time, 1=Delayed, 2=Canceled, 3=Diverted
    when(col("Cancelled").cast("int") == 1, 2)
    .when(col("Diverted").cast("int") == 1, 3)
    .when(col("DepDelay").cast("double").cast("int") > 15, 1)
    .otherwise(0).alias("flight_status")
)

In [0]:
display(silver_flights.limit(10))

In [0]:
spark.sql("DROP TABLE IF EXISTS aviation_project.silver_historical_flights")

In [0]:
# ---------------------------------------------------------
# Part 4: Write and Optimize
# ---------------------------------------------------------
print("5. Writing to Silver Delta Table...")
silver_flights.write.format("delta").mode("overwrite").saveAsTable("aviation_project.silver_historical_flights")

In [0]:
spark.sql("OPTIMIZE aviation_project.silver_historical_flights ZORDER BY (scheduled_time_utc)")

final_count = spark.table("aviation_project.silver_historical_flights").count()
print(f"6. SUCCESS! Processed {final_count} historical records (COVID years excluded) into the new architecture.")